In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import yaml
import numpy as np
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/atlas-go-revision"
os.chdir(PROJECT_ROOT)

with open("configs/config.yaml", "r") as file:
    cfg = yaml.safe_load(file)

RAW_DIR = cfg["paths"]["data_raw"]
PROCESSED_DIR = cfg["paths"]["data_processed"]
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Main split:", cfg["split"]["main_split"])
print("Rare GO-term threshold:", cfg["data"]["min_proteins_per_go"])

Mounted at /content/drive
Project: /content/drive/MyDrive/atlas-go-revision
Main split: homology
Rare GO-term threshold: 5


In [2]:
!pip install goatools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 9.7 MB/s eta 0:00:00


In [3]:
import re
import json
import random
import shutil
import subprocess
from collections import Counter

from goatools.obo_parser import GODag
from sklearn.model_selection import train_test_split

SEED = cfg["split"]["random_seed_for_split"]

random.seed(SEED)
np.random.seed(SEED)

S_COLS = cfg["data"]["static_features"]
D_COLS = cfg["data"]["dynamic_features"]

print("Imports complete.")

Imports complete.


In [4]:
atlas_path = f"{RAW_DIR}/ATLAS.csv"
go_path = f"{RAW_DIR}/sequence_GoLabels.csv"

atlas = pd.read_csv(atlas_path)
go_data = pd.read_csv(go_path)

# Clean any invisible spaces from ATLAS column names.
atlas.columns = atlas.columns.str.replace("\xa0", " ", regex=False).str.strip()
go_data.columns = go_data.columns.str.strip()

print("ATLAS shape:", atlas.shape)
print("GO data shape:", go_data.shape)

required_atlas_columns = ["PDB", "UniProt"] + S_COLS + D_COLS
required_go_columns = ["UniProt", "Sequence", "MF", "BP", "CC"]

missing_atlas = [col for col in required_atlas_columns if col not in atlas.columns]
missing_go = [col for col in required_go_columns if col not in go_data.columns]

if missing_atlas:
    raise ValueError(f"Missing ATLAS columns: {missing_atlas}")

if missing_go:
    raise ValueError(f"Missing GO-data columns: {missing_go}")

# Keep one ATLAS record per UniProt protein.
atlas = atlas[required_atlas_columns].copy()
n_atlas_original = len(atlas)

atlas = atlas.drop_duplicates(subset="UniProt", keep="first")
n_after_duplicates = len(atlas)

# Static features are essential for S/PS/ES/PES feature sets.
atlas = atlas.dropna(subset=["UniProt"] + S_COLS)
n_after_static_missing = len(atlas)

# Do NOT impute dynamic values here.
# Dynamic-feature imputation will use the training-set mean in Notebook 02.
merged = pd.merge(atlas, go_data, on="UniProt", how="inner")
n_merged = len(merged)

print("\nDataset audit")
print("-" * 50)
print("ATLAS rows originally:                 ", n_atlas_original)
print("After duplicate UniProt removal:       ", n_after_duplicates)
print("After missing static-feature removal:  ", n_after_static_missing)
print("After ATLAS-GO merge:                  ", n_merged)

ATLAS shape: (1938, 15)
GO data shape: (1510, 5)

Dataset audit
--------------------------------------------------
ATLAS rows originally:                  1938
After duplicate UniProt removal:        1534
After missing static-feature removal:   1534
After ATLAS-GO merge:                   1504


Cell 4 — Parse direct GO annotations

In [5]:
def extract_go_ids(text):
    """Extract unique GO:XXXXXXX IDs from one cell."""
    if pd.isna(text) or str(text).strip() == "":
        return []

    return sorted(set(re.findall(r"GO:\d{7}", str(text))))


df = merged.copy().reset_index(drop=True)

df["MFO_direct"] = df["MF"].apply(extract_go_ids)
df["BPO_direct"] = df["BP"].apply(extract_go_ids)
df["CCO_direct"] = df["CC"].apply(extract_go_ids)

df["MFO_direct_count"] = df["MFO_direct"].apply(len)
df["BPO_direct_count"] = df["BPO_direct"].apply(len)
df["CCO_direct_count"] = df["CCO_direct"].apply(len)

df["total_direct_count"] = (
    df["MFO_direct_count"]
    + df["BPO_direct_count"]
    + df["CCO_direct_count"]
)

print("Proteins after merge:", len(df))
print("\nDirect GO annotations per protein:")
print(df[
    ["MFO_direct_count", "BPO_direct_count", "CCO_direct_count", "total_direct_count"]
].describe())

Proteins after merge: 1504

Direct GO annotations per protein:
       MFO_direct_count  BPO_direct_count  CCO_direct_count  \
count       1504.000000       1504.000000        1504.00000   
mean           2.767952          4.212101           2.31117   
std            3.208233          9.414861           3.70811   
min            0.000000          0.000000           0.00000   
25%            1.000000          0.000000           0.00000   
50%            2.000000          1.000000           1.00000   
75%            4.000000          4.000000           3.00000   
max           36.000000        162.000000          39.00000   

       total_direct_count  
count         1504.000000  
mean             9.291223  
std             14.619104  
min              0.000000  
25%              2.000000  
50%              5.000000  
75%             10.000000  
max            191.000000  


Cell 5 — Load GO hierarchy and propagate annotations to ancestors

In [6]:
obo_path = cfg["go_hierarchy"]["obo_path"]

if not os.path.exists(obo_path):
    raise FileNotFoundError(
        f"GO hierarchy file not found:\n{obo_path}\n"
        "Upload go-basic.obo to data/raw and run again."
    )

go_dag = GODag(obo_path, optional_attrs={"relationship"})
print("GO ontology loaded:", len(go_dag), "terms")

def add_go_ancestors(go_ids):
    """
    Apply GO true-path propagation using both is_a and part_of relations.
    """
    ancestors = set()
    to_visit = list(go_ids)
    visited = set()

    while to_visit:
        current_id = to_visit.pop()

        if current_id in visited or current_id not in go_dag:
            continue

        visited.add(current_id)
        term = go_dag[current_id]

        # is_a parents
        parent_ids = [parent.item_id for parent in term.parents]

        # part_of parents
        for parent in term.relationship.get("part_of", set()):
            parent_ids.append(parent.item_id)

        for parent_id in parent_ids:
            if parent_id not in ancestors:
                ancestors.add(parent_id)
                to_visit.append(parent_id)

    return sorted(set(go_ids) | ancestors)

if cfg["go_hierarchy"]["enabled"] and cfg["go_hierarchy"]["propagate_labels"]:
    df["MFO_clean"] = df["MFO_direct"].apply(add_go_ancestors)
    df["BPO_clean"] = df["BPO_direct"].apply(add_go_ancestors)
    df["CCO_clean"] = df["CCO_direct"].apply(add_go_ancestors)
    print("GO true-path propagation applied to labels.")
else:
    df["MFO_clean"] = df["MFO_direct"]
    df["BPO_clean"] = df["BPO_direct"]
    df["CCO_clean"] = df["CCO_direct"]
    print("GO hierarchy propagation is disabled.")

/content/drive/MyDrive/atlas-go-revision/data/raw/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms; optional_attrs(relationship)
GO ontology loaded: 42666 terms
GO true-path propagation applied to labels.


Cell 6 — Check that propagated terms remain in their correct ontology

In [7]:
namespace_map = {
    "molecular_function": "MFO",
    "biological_process": "BPO",
    "cellular_component": "CCO",
}

def keep_expected_namespace(go_ids, expected_namespace):
    """Keep only GO terms that belong to the expected GO namespace."""
    kept = []

    for go_id in go_ids:
        if go_id in go_dag:
            namespace = namespace_map.get(go_dag[go_id].namespace)

            if namespace == expected_namespace:
                kept.append(go_id)

    return sorted(set(kept))

df["MFO_clean"] = df["MFO_clean"].apply(
    lambda terms: keep_expected_namespace(terms, "MFO")
)

df["BPO_clean"] = df["BPO_clean"].apply(
    lambda terms: keep_expected_namespace(terms, "BPO")
)

df["CCO_clean"] = df["CCO_clean"].apply(
    lambda terms: keep_expected_namespace(terms, "CCO")
)

df["MFO_count_before_term_filter"] = df["MFO_clean"].apply(len)
df["BPO_count_before_term_filter"] = df["BPO_clean"].apply(len)
df["CCO_count_before_term_filter"] = df["CCO_clean"].apply(len)

df["total_count_before_term_filter"] = (
    df["MFO_count_before_term_filter"]
    + df["BPO_count_before_term_filter"]
    + df["CCO_count_before_term_filter"]
)

print("GO annotations after hierarchy propagation:")
print(df[
    [
        "MFO_count_before_term_filter",
        "BPO_count_before_term_filter",
        "CCO_count_before_term_filter",
        "total_count_before_term_filter",
    ]
].describe())

GO annotations after hierarchy propagation:
       MFO_count_before_term_filter  BPO_count_before_term_filter  \
count                   1504.000000                   1504.000000   
mean                      12.468085                     32.183511   
std                       12.099072                     57.084833   
min                        0.000000                      0.000000   
25%                        4.000000                      0.000000   
50%                       10.000000                     14.000000   
75%                       18.000000                     35.000000   
max                       88.000000                    749.000000   

       CCO_count_before_term_filter  total_count_before_term_filter  
count                   1504.000000                     1504.000000  
mean                       9.347074                       53.998670  
std                       12.935554                       73.620469  
min                        0.000000                   

Cell 7 — Create CD-HIT FASTA and run CD-HIT

In [9]:
# CD-HIT creates the homology-aware split.
!apt-get -qq update
!apt-get -qq install -y cd-hit

print("Packages and CD-HIT are ready.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package cd-hit.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../cd-hit_4.8.1-4_amd64.deb ...
Unpacking cd-hit (4.8.1-4) ...
Setting up cd-hit (4.8.1-4) ...
Processing triggers for man-db (2.10.2-1) ...
Packages and CD-HIT are ready.


In [10]:
# We create clusters BEFORE choosing the final GO vocabulary.
# This ensures the test set cannot decide which GO terms are retained.

fasta_path = f"{PROCESSED_DIR}/proteins_for_cdhit.fasta"

with open(fasta_path, "w") as file:
    for _, row in df.iterrows():
        protein_id = row["UniProt"]

        sequence = str(row["Sequence"]).upper().replace(" ", "")
        sequence = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", sequence)

        if len(sequence) == 0:
            raise ValueError(f"Empty sequence after cleaning: {protein_id}")

        file.write(f">{protein_id}\n{sequence}\n")

cdhit_binary = cfg["homology_split"]["cdhit_binary"]
identity_threshold = cfg["homology_split"]["identity_threshold"]
word_size = cfg["homology_split"]["word_size"]

if shutil.which(cdhit_binary) is None:
    raise FileNotFoundError(
        "CD-HIT was not found. Re-run the package-installation cell in 00_SETUP."
    )

cdhit_output = f"{PROCESSED_DIR}/proteins_cdhit"

command = [
    cdhit_binary,
    "-i", fasta_path,
    "-o", cdhit_output,
    "-c", str(identity_threshold),
    "-n", str(word_size),
    "-d", "0",
    "-T", "2",
    "-M", "0",
]

print("Running CD-HIT:")
print(" ".join(command))

subprocess.run(command, check=True)

cluster_file = f"{cdhit_output}.clstr"

if not os.path.exists(cluster_file):
    raise FileNotFoundError("CD-HIT cluster file was not created.")

print("CD-HIT clustering complete.")

Running CD-HIT:
cd-hit -i /content/drive/MyDrive/atlas-go-revision/data/processed/proteins_for_cdhit.fasta -o /content/drive/MyDrive/atlas-go-revision/data/processed/proteins_cdhit -c 0.4 -n 2 -d 0 -T 2 -M 0
CD-HIT clustering complete.


Cell 8 — Read the CD-HIT clusters

In [11]:
def read_cdhit_clusters(cluster_file):
    """Create a mapping: UniProt ID -> CD-HIT cluster ID."""
    cluster_of = {}
    current_cluster = None

    with open(cluster_file, "r") as file:
        for line in file:
            line = line.strip()

            if line.startswith(">Cluster"):
                current_cluster = int(line.split()[-1])

            elif line:
                match = re.search(r">(.+?)\.\.\.", line)

                if match:
                    protein_id = match.group(1)
                    cluster_of[protein_id] = current_cluster

    return cluster_of


cluster_of = read_cdhit_clusters(cluster_file)

missing_ids = set(df["UniProt"]) - set(cluster_of)

if missing_ids:
    raise ValueError(
        f"{len(missing_ids)} proteins are missing CD-HIT cluster assignments."
    )

df["cluster_id"] = df["UniProt"].map(cluster_of)

cluster_sizes = df.groupby("cluster_id").size().sort_values(ascending=False)

print("Proteins:", len(df))
print("CD-HIT clusters:", len(cluster_sizes))
print("Largest cluster:", cluster_sizes.iloc[0])
print("Median cluster size:", cluster_sizes.median())

Proteins: 1504
CD-HIT clusters: 1388
Largest cluster: 7
Median cluster size: 1.0


Cell 9 — Make the CD-HIT homology-aware split

In [12]:
def make_homology_aware_split(dataframe, train_ratio, val_ratio, seed):
    """
    Put every complete CD-HIT cluster into exactly one partition.
    No cluster can appear in both train and test.
    """
    rng = random.Random(seed)

    cluster_to_indices = (
        dataframe.groupby("cluster_id")
        .apply(lambda group: list(group.index))
        .to_dict()
    )

    clusters = list(cluster_to_indices.items())

    # Assign the largest clusters first for better split-size balance.
    clusters = sorted(clusters, key=lambda item: len(item[1]), reverse=True)

    # Randomize ties reproducibly.
    rng.shuffle(clusters)
    clusters = sorted(clusters, key=lambda item: len(item[1]), reverse=True)

    n_total = len(dataframe)

    target_sizes = {
        "train": n_total * train_ratio,
        "validation": n_total * val_ratio,
        "test": n_total * (1 - train_ratio - val_ratio),
    }

    split_indices = {
        "train": [],
        "validation": [],
        "test": [],
    }

    split_sizes = {
        "train": 0,
        "validation": 0,
        "test": 0,
    }

    for _, indices in clusters:
        deficits = {
            split_name: target_sizes[split_name] - split_sizes[split_name]
            for split_name in split_sizes
        }

        destination = max(deficits, key=deficits.get)

        split_indices[destination].extend(indices)
        split_sizes[destination] += len(indices)

    return split_indices


initial_split = make_homology_aware_split(
    dataframe=df,
    train_ratio=cfg["data"]["train_ratio"],
    val_ratio=cfg["data"]["val_ratio"],
    seed=SEED,
)

# Record the split before filtering GO terms.
df["split"] = ""

for split_name, indices in initial_split.items():
    df.loc[indices, "split"] = split_name

if (df["split"] == "").any():
    raise ValueError("Some proteins were not assigned to a split.")

print(df["split"].value_counts())

split
train         1203
validation     151
test           150
Name: count, dtype: int64


/tmp/ipykernel_2453/1113487094.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda group: list(group.index))


Cell 10 — Verify CD-HIT cluster separation

In [13]:
train_clusters = set(df.loc[df["split"] == "train", "cluster_id"])
val_clusters = set(df.loc[df["split"] == "validation", "cluster_id"])
test_clusters = set(df.loc[df["split"] == "test", "cluster_id"])

assert train_clusters.isdisjoint(val_clusters)
assert train_clusters.isdisjoint(test_clusters)
assert val_clusters.isdisjoint(test_clusters)

print("✓ No CD-HIT cluster is shared across train, validation, or test.")
print(f"Train clusters:      {len(train_clusters)}")
print(f"Validation clusters: {len(val_clusters)}")
print(f"Test clusters:       {len(test_clusters)}")

✓ No CD-HIT cluster is shared across train, validation, or test.
Train clusters:      1087
Validation clusters: 151
Test clusters:       150


Cell 11 — Select the GO vocabulary from training proteins only

In [14]:
# Important leakage safeguard:
# Count GO-term frequency using TRAINING proteins only.

MIN_PROTEINS_PER_GO = cfg["data"]["min_proteins_per_go"]

train_df = df[df["split"] == "train"].copy()

mfo_train_counts = Counter(
    go_id
    for terms in train_df["MFO_clean"]
    for go_id in terms
)

bpo_train_counts = Counter(
    go_id
    for terms in train_df["BPO_clean"]
    for go_id in terms
)

cco_train_counts = Counter(
    go_id
    for terms in train_df["CCO_clean"]
    for go_id in terms
)

mfo_vocab = sorted([
    go_id for go_id, count in mfo_train_counts.items()
    if count >= MIN_PROTEINS_PER_GO
])

bpo_vocab = sorted([
    go_id for go_id, count in bpo_train_counts.items()
    if count >= MIN_PROTEINS_PER_GO
])

cco_vocab = sorted([
    go_id for go_id, count in cco_train_counts.items()
    if count >= MIN_PROTEINS_PER_GO
])

mfo_set = set(mfo_vocab)
bpo_set = set(bpo_vocab)
cco_set = set(cco_vocab)

go_vocab = mfo_vocab + bpo_vocab + cco_vocab

print("GO vocabulary selected from training proteins only")
print("-" * 55)
print("MFO terms:", len(mfo_vocab))
print("BPO terms:", len(bpo_vocab))
print("CCO terms:", len(cco_vocab))
print("Total GO terms:", len(go_vocab))

GO vocabulary selected from training proteins only
-------------------------------------------------------
MFO terms: 344
BPO terms: 1350
CCO terms: 267
Total GO terms: 1961


Cell 12 — Apply the fixed training vocabulary to every split

In [15]:
# Validation and test labels use the fixed vocabulary selected from training.
# They cannot introduce additional GO terms.

df["MFO_retained"] = df["MFO_clean"].apply(
    lambda terms: sorted(go_id for go_id in terms if go_id in mfo_set)
)

df["BPO_retained"] = df["BPO_clean"].apply(
    lambda terms: sorted(go_id for go_id in terms if go_id in bpo_set)
)

df["CCO_retained"] = df["CCO_clean"].apply(
    lambda terms: sorted(go_id for go_id in terms if go_id in cco_set)
)

df["MFO_count_after_term_filter"] = df["MFO_retained"].apply(len)
df["BPO_count_after_term_filter"] = df["BPO_retained"].apply(len)
df["CCO_count_after_term_filter"] = df["CCO_retained"].apply(len)

df["total_count_after_term_filter"] = (
    df["MFO_count_after_term_filter"]
    + df["BPO_count_after_term_filter"]
    + df["CCO_count_after_term_filter"]
)

n_before_zero_label_removal = len(df)

# No "minimum five GO terms per protein" rule.
# Remove only proteins that have zero labels after applying training vocabulary.
df = df[df["total_count_after_term_filter"] > 0].copy()

n_zero_label_removed = n_before_zero_label_removal - len(df)

print("Proteins before zero-label removal:", n_before_zero_label_removal)
print("Proteins removed with zero retained labels:", n_zero_label_removed)
print("Final proteins:", len(df))

print("\nFinal proteins by partition:")
print(df["split"].value_counts())

Proteins before zero-label removal: 1504
Proteins removed with zero retained labels: 148
Final proteins: 1356

Final proteins by partition:
split
train         1091
validation     133
test           132
Name: count, dtype: int64


Cell 13 — Rebuild final indices and validate the split

In [16]:
# Reset row indices after removing zero-label proteins.
df = df.reset_index(drop=True)

train_idx = np.where(df["split"].to_numpy() == "train")[0]
val_idx = np.where(df["split"].to_numpy() == "validation")[0]
test_idx = np.where(df["split"].to_numpy() == "test")[0]

# Verify no cluster crossing remains after zero-label removal.
train_clusters = set(df.iloc[train_idx]["cluster_id"])
val_clusters = set(df.iloc[val_idx]["cluster_id"])
test_clusters = set(df.iloc[test_idx]["cluster_id"])

assert train_clusters.isdisjoint(val_clusters)
assert train_clusters.isdisjoint(test_clusters)
assert val_clusters.isdisjoint(test_clusters)

print("Final homology-aware split")
print("-" * 50)

for split_name, indices in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Test", test_idx),
]:
    print(
        f"{split_name:12s}: "
        f"{len(indices):4d} proteins "
        f"({100 * len(indices) / len(df):.1f}%)"
    )

print("\n✓ No CD-HIT cluster leakage after filtering.")

Final homology-aware split
--------------------------------------------------
Train       : 1091 proteins (80.5%)
Validation  :  133 proteins (9.8%)
Test        :  132 proteins (9.7%)

✓ No CD-HIT cluster leakage after filtering.


Cell 14 — Build the final label matrix

In [17]:
go_to_index = {
    go_id: index
    for index, go_id in enumerate(go_vocab)
}

Y_all = np.zeros(
    (len(df), len(go_vocab)),
    dtype=np.float32,
)

for row_index, row in df.iterrows():
    retained_terms = (
        row["MFO_retained"]
        + row["BPO_retained"]
        + row["CCO_retained"]
    )

    for go_id in retained_terms:
        Y_all[row_index, go_to_index[go_id]] = 1.0

# This must always be true after Cell 12.
assert (Y_all.sum(axis=1) > 0).all()

print("Y_all shape:", Y_all.shape)
print("Mean labels per protein:", round(Y_all.sum(axis=1).mean(), 2))

np.save(f"{PROCESSED_DIR}/Y_all.npy", Y_all)

Y_all shape: (1356, 1961)
Mean labels per protein: 50.39


Cell 15 — Save vocabulary, prevalence, and filtering reports

In [18]:
go_namespace_map = pd.DataFrame({
    "index": range(len(go_vocab)),
    "go_id": go_vocab,
    "namespace": (
        ["MFO"] * len(mfo_vocab)
        + ["BPO"] * len(bpo_vocab)
        + ["CCO"] * len(cco_vocab)
    ),
})

go_namespace_map.to_csv(
    f"{PROCESSED_DIR}/go_namespace_map.csv",
    index=False,
)

ontology_slices = {
    "MFO": {
        "start": 0,
        "end": len(mfo_vocab),
    },
    "BPO": {
        "start": len(mfo_vocab),
        "end": len(mfo_vocab) + len(bpo_vocab),
    },
    "CCO": {
        "start": len(mfo_vocab) + len(bpo_vocab),
        "end": len(go_vocab),
    },
    "total_go_terms": len(go_vocab),
}

with open(f"{PROCESSED_DIR}/go_ontology_slices.yaml", "w") as file:
    yaml.dump(ontology_slices, file, sort_keys=False)

# Count positive proteins in each partition.
class_prevalence = go_namespace_map.copy()

class_prevalence["n_train_proteins"] = Y_all[train_idx].sum(axis=0).astype(int)
class_prevalence["n_validation_proteins"] = Y_all[val_idx].sum(axis=0).astype(int)
class_prevalence["n_test_proteins"] = Y_all[test_idx].sum(axis=0).astype(int)

class_prevalence.to_csv(
    f"{PROCESSED_DIR}/class_prevalence_by_split.csv",
    index=False,
)
assert (class_prevalence["n_train_proteins"] >= MIN_PROTEINS_PER_GO).all()

print("✓ Every retained GO term occurs in at least 5 training proteins.")

filtering_audit = pd.DataFrame([
    {
        "stage": "ATLAS rows before duplicate removal",
        "n_proteins": n_atlas_original,
    },
    {
        "stage": "ATLAS rows after duplicate removal",
        "n_proteins": n_after_duplicates,
    },
    {
        "stage": "ATLAS rows after missing static-feature removal",
        "n_proteins": n_after_static_missing,
    },
    {
        "stage": "ATLAS-UniProt GO intersection before split",
        "n_proteins": n_merged,
    },
    {
        "stage": "After training-derived GO vocabulary applied",
        "n_proteins": n_before_zero_label_removal,
    },
    {
        "stage": "Final proteins after zero-label removal",
        "n_proteins": len(df),
    },
])

filtering_audit.to_csv(
    f"{PROCESSED_DIR}/filtering_audit.csv",
    index=False,
)

print("Saved vocabulary, prevalence, and filtering reports.")


✓ Every retained GO term occurs in at least 5 training proteins.
Saved vocabulary, prevalence, and filtering reports.


In [19]:
annotation_distribution = df[
    [
        "UniProt",
        "split",
        "MFO_direct_count",
        "BPO_direct_count",
        "CCO_direct_count",
        "total_direct_count",
        "MFO_count_before_term_filter",
        "BPO_count_before_term_filter",
        "CCO_count_before_term_filter",
        "total_count_before_term_filter",
        "MFO_count_after_term_filter",
        "BPO_count_after_term_filter",
        "CCO_count_after_term_filter",
        "total_count_after_term_filter",
    ]
].copy()

annotation_distribution.to_csv(
    f"{PROCESSED_DIR}/annotation_distribution.csv",
    index=False,
)

annotation_summary = (
    annotation_distribution
    .groupby("split")
    [
        [
            "MFO_direct_count",
            "BPO_direct_count",
            "CCO_direct_count",
            "total_direct_count",
            "MFO_count_before_term_filter",
            "BPO_count_before_term_filter",
            "CCO_count_before_term_filter",
            "total_count_before_term_filter",
            "MFO_count_after_term_filter",
            "BPO_count_after_term_filter",
            "CCO_count_after_term_filter",
            "total_count_after_term_filter",
        ]
    ]
    .describe()
)

annotation_summary.to_csv(
    f"{PROCESSED_DIR}/annotation_distribution_summary_by_split.csv"
)

print("Saved annotation distribution reports.")

Saved annotation distribution reports.


Cell 16 — Save split IDs and cluster report

In [20]:
np.savez(
    f"{PROCESSED_DIR}/splits_homology.npz",
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
)

for split_name, indices in [
    ("train", train_idx),
    ("val", val_idx),
    ("test", test_idx),
]:
    df.iloc[indices][["UniProt", "cluster_id"]].to_csv(
        f"{PROCESSED_DIR}/{split_name}_ids_homology.csv",
        index=False,
    )

cluster_report = (
    df.groupby("cluster_id")
    .agg(
        n_proteins=("UniProt", "size"),
        split=("split", "first"),
        representative_protein=("UniProt", "first"),
    )
    .reset_index()
    .sort_values("n_proteins", ascending=False)
)

cluster_report.to_csv(
    f"{PROCESSED_DIR}/cdhit_cluster_report.csv",
    index=False,
)

split_report = pd.DataFrame([
    {
        "split": "train",
        "n_proteins": len(train_idx),
        "n_clusters": len(train_clusters),
        "percentage": round(100 * len(train_idx) / len(df), 2),
    },
    {
        "split": "validation",
        "n_proteins": len(val_idx),
        "n_clusters": len(val_clusters),
        "percentage": round(100 * len(val_idx) / len(df), 2),
    },
    {
        "split": "test",
        "n_proteins": len(test_idx),
        "n_clusters": len(test_clusters),
        "percentage": round(100 * len(test_idx) / len(df), 2),
    },
])

split_report.to_csv(
    f"{PROCESSED_DIR}/homology_split_report.csv",
    index=False,
)

print(split_report)

        split  n_proteins  n_clusters  percentage
0       train        1091         979       80.46
1  validation         133         133        9.81
2        test         132         132        9.73


Cell 17 — Truncation report

In [21]:
PROTBERT_MAX = cfg["data"]["protbert_max_residues"]
ESM2_MAX = cfg["data"]["esm2_max_residues"]

truncation_report = pd.DataFrame({
    "UniProt": df["UniProt"],
    "split": df["split"],
    "sequence_length": df["Sequence"].astype(str).str.len(),
})

truncation_report["truncated_protbert"] = (
    truncation_report["sequence_length"] > PROTBERT_MAX
)

truncation_report["truncated_esm2"] = (
    truncation_report["sequence_length"] > ESM2_MAX
)

truncation_report.to_csv(
    f"{PROCESSED_DIR}/truncation_report_by_protein.csv",
    index=False,
)

truncation_summary = (
    truncation_report
    .groupby("split")
    .agg(
        n_proteins=("UniProt", "size"),
        protbert_truncated=("truncated_protbert", "sum"),
        esm2_truncated=("truncated_esm2", "sum"),
        mean_sequence_length=("sequence_length", "mean"),
        max_sequence_length=("sequence_length", "max"),
    )
    .reset_index()
)

truncation_summary.to_csv(
    f"{PROCESSED_DIR}/truncation_summary_by_split.csv",
    index=False,
)

print(truncation_summary)

        split  n_proteins  protbert_truncated  esm2_truncated  \
0        test         132                  16              16   
1       train        1091                  93              93   
2  validation         133                  14              14   

   mean_sequence_length  max_sequence_length  
0            503.818182                 3097  
1            517.472044                 7096  
2            555.293233                 7176  


Cell 18 — Save final processed data

In [22]:
df_to_save = df.copy()

list_columns = [
    "MFO_direct", "BPO_direct", "CCO_direct",
    "MFO_clean", "BPO_clean", "CCO_clean",
    "MFO_retained", "BPO_retained", "CCO_retained",
]

for column in list_columns:
    df_to_save[column] = df_to_save[column].apply(
        lambda terms: ";".join(terms)
    )

df_to_save.to_csv(
    f"{PROCESSED_DIR}/full_dataset_processed.csv",
    index=False,
)

print("=" * 60)
print("LEAKAGE-SAFE DATA PREPARATION COMPLETE")
print("=" * 60)
print("Final proteins:", len(df))
print("Final GO terms, selected from train only:", len(go_vocab))
print("MFO / BPO / CCO:", len(mfo_vocab), "/", len(bpo_vocab), "/", len(cco_vocab))
print("Y_all shape:", Y_all.shape)
print("Main split: CD-HIT homology-aware split")
print("No CD-HIT cluster is shared across partitions.")

LEAKAGE-SAFE DATA PREPARATION COMPLETE
Final proteins: 1356
Final GO terms, selected from train only: 1961
MFO / BPO / CCO: 344 / 1350 / 267
Y_all shape: (1356, 1961)
Main split: CD-HIT homology-aware split
No CD-HIT cluster is shared across partitions.
